**Hyperparameter Tuning**

In [10]:
from sklearn.model_selection import GridSearchCV,RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,ConfusionMatrixDisplay,classification_report,roc_auc_score
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

Load the models

In [4]:
rf_model = joblib.load('/content/random_forest_rfe.pkl')
dt_model = joblib.load('/content/decision_tree_rfe.pkl')
lr_model = joblib.load('/content/logistic_regression_rfe.pkl')
svm_model = joblib.load('/content/svm_rfe.pkl')
knn_model = joblib.load('/content/kmeans_model.pkl')
pca_model = joblib.load('/content/pca_model.pkl')

In [5]:
df = pd.read_csv('/content/features_selected_rfe.csv')

In [6]:
X = df.drop('num',axis=1)
y = df['num']

In [7]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [12]:
def print_performance(model,X_train,y_train,X_test,y_test):
    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    print(f'Train Accuracy: {accuracy_score(y_train,model.predict(X_train))*100:.2f}%')
    print(f'Test Accuracy: {accuracy_score(y_test,y_pred)*100:.2f}%')
    print(f"ROC score: {roc_auc_score(y_test,y_pred):.3f}")
    print(f"Precision score: {precision_score(y_test,y_pred):.2f}")
    print(f"Recall score: {recall_score(y_test,y_pred):.2f}")
    print(f"F1 Score score: {f1_score(y_test,y_pred):.2f}")

**GridSearchCV**



Logistic Regression

In [13]:
param_grid = {
    'max_iter': [1,10,100,1000,10000,100000]

}
grid_search = GridSearchCV(LogisticRegression(), param_grid, cv=5)
grid_search.fit(X_train, y_train)

print("Best Hyperparameters:", grid_search.best_params_)

Best Hyperparameters: {'max_iter': 10}


In [15]:
b_lr = LogisticRegression(max_iter=10)
print_performance(b_lr,X_train,y_train,X_test,y_test)

Train Accuracy: 84.30%
Test Accuracy: 88.52%
ROC score: 0.886
Precision score: 0.90
Recall score: 0.88
F1 Score score: 0.89


Desicion Tree

In [16]:
# Define the parameter grid for GridSearchCV
param_grid_dt = {
    'max_depth': [None, 10, 20, 30, 40, 50],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'criterion': ['gini', 'entropy']
}

# Initialize a new Decision Tree Classifier instance
dt_classifier = DecisionTreeClassifier(random_state=42) # Added random_state for reproducibility

# Assuming X_train and y_train are defined from your data loading step:
# Initialize GridSearchCV
grid_search_dt = GridSearchCV(estimator=dt_classifier, param_grid=param_grid_dt, cv=5, scoring='recall', n_jobs=-1)

# Fit GridSearchCV
grid_search_dt.fit(X_train, y_train)

# Print the best parameters and best score
print("Best parameters found for Decision Tree: ", grid_search_dt.best_params_)
print("Best accuracy score found for Decision Tree: ", grid_search_dt.best_score_)

# You can then access the best model with grid_search_dt.best_estimator_

Best parameters found for Decision Tree:  {'criterion': 'entropy', 'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 10}
Best accuracy score found for Decision Tree:  0.7012987012987014


In [17]:
b_dt = DecisionTreeClassifier(criterion='entropy', min_samples_leaf=2, min_samples_split=10)
print_performance(b_dt,X_train,y_train,X_test,y_test)

Train Accuracy: 91.32%
Test Accuracy: 77.05%
ROC score: 0.775
Precision score: 0.85
Recall score: 0.69
F1 Score score: 0.76


Random Forest

In [18]:
param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}


rf_classifier = RandomForestClassifier(random_state=42)
grid_search_rf = GridSearchCV(estimator=rf_classifier, param_grid=param_grid_rf, cv=5, scoring='recall', n_jobs=-1)

# Fit GridSearchCV
grid_search_rf.fit(X_train, y_train)

# Print the best parameters and best score
print("Best parameters found for Random Forest: ", grid_search_rf.best_params_)
print("Best recall score found for Random Forest: ", grid_search_rf.best_score_)



Best parameters found for Random Forest:  {'bootstrap': False, 'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}
Best recall score found for Random Forest:  0.7480519480519481


In [19]:
b_rf = RandomForestClassifier(n_estimators=200, min_samples_split=2, min_samples_leaf=2, max_depth=None, bootstrap=False)
print_performance(b_rf,X_train,y_train,X_test,y_test)

Train Accuracy: 99.59%
Test Accuracy: 88.52%
ROC score: 0.887
Precision score: 0.93
Recall score: 0.84
F1 Score score: 0.89


SVM

In [20]:
param_grid_svm = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1],
    'kernel': ['linear', 'rbf', 'poly', 'sigmoid']
}


svm_classifier = SVC(random_state=42)
grid_search_svm = GridSearchCV(estimator=svm_classifier, param_grid=param_grid_svm, cv=5, scoring='recall', n_jobs=-1)


grid_search_svm.fit(X_train, y_train)


print("Best parameters found for SVM: ", grid_search_svm.best_params_)
print("Best recall score found for SVM: ", grid_search_svm.best_score_)

Best parameters found for SVM:  {'C': 100, 'gamma': 'auto', 'kernel': 'sigmoid'}
Best recall score found for SVM:  0.7649350649350649


In [21]:
b_svm = SVC(C=100, gamma='auto', kernel='sigmoid')
print_performance(b_svm,X_train,y_train,X_test,y_test)

Train Accuracy: 74.38%
Test Accuracy: 78.69%
ROC score: 0.789
Precision score: 0.83
Recall score: 0.75
F1 Score score: 0.79


**RandomizedSearchCV**

In [22]:
param_dist_lr = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear', 'saga']
}

lr_classifier = LogisticRegression(random_state=42)
random_search_lr = RandomizedSearchCV(estimator=lr_classifier, param_distributions=param_dist_lr, n_iter=100, cv=5, scoring='recall', random_state=42, n_jobs=-1)
random_search_lr.fit(X_train, y_train)

print("Best parameters found for Logistic Regression (Randomized Search): ", random_search_lr.best_params_)
print("Best recall score found for Logistic Regression (Randomized Search): ", random_search_lr.best_score_)



Best parameters found for Logistic Regression (Randomized Search):  {'solver': 'liblinear', 'penalty': 'l2', 'C': 0.001}
Best recall score found for Logistic Regression (Randomized Search):  0.7757575757575758


In [23]:
lr_rs = LogisticRegression(C=0.001, penalty='l2', solver='liblinear')
print_performance(lr_rs,X_train,y_train,X_test,y_test)

Train Accuracy: 82.23%
Test Accuracy: 83.61%
ROC score: 0.839
Precision score: 0.89
Recall score: 0.78
F1 Score score: 0.83


Discion Tree

In [24]:
param_dist_dt = {
    'max_depth': [None, 10, 20, 30, 40, 50],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'criterion': ['gini', 'entropy']
}

dt_classifier = DecisionTreeClassifier(random_state=42)
random_search_dt = RandomizedSearchCV(estimator=dt_classifier, param_distributions=param_dist_dt, n_iter=100, cv=5, scoring='accuracy', random_state=42, n_jobs=-1)

random_search_dt.fit(X_train, y_train)
print("Best parameters found for Decision Tree (Randomized Search): ", random_search_dt.best_params_)
print("Best accuracy score found for Decision Tree (Randomized Search): ", random_search_dt.best_score_)

Best parameters found for Decision Tree (Randomized Search):  {'min_samples_split': 20, 'min_samples_leaf': 8, 'max_depth': 20, 'criterion': 'entropy'}
Best accuracy score found for Decision Tree (Randomized Search):  0.7807823129251701


In [26]:
dt_rs = DecisionTreeClassifier(criterion='entropy', min_samples_leaf=8, min_samples_split=20, max_depth=20)
print_performance(dt_rs,X_train,y_train,X_test,y_test)

Train Accuracy: 84.30%
Test Accuracy: 86.89%
ROC score: 0.870
Precision score: 0.90
Recall score: 0.84
F1 Score score: 0.87


Random Forest

In [25]:
param_dist_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}
rf_classifier = RandomForestClassifier(random_state=42)
random_search_rf = RandomizedSearchCV(estimator=rf_classifier, param_distributions=param_dist_rf, n_iter=100, cv=5, scoring='recall', random_state=42, n_jobs=-1)


random_search_rf.fit(X_train, y_train)


print("Best parameters found for Random Forest (Randomized Search): ", random_search_rf.best_params_)
print("Best recall score found for Random Forest (Randomized Search): ", random_search_rf.best_score_)


Best parameters found for Random Forest (Randomized Search):  {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': 20, 'bootstrap': False}
Best recall score found for Random Forest (Randomized Search):  0.7480519480519481


In [27]:
rf_rs = RandomForestClassifier(n_estimators=200, min_samples_split=2, min_samples_leaf=2, max_depth=20, bootstrap=False)
print_performance(rf_rs,X_train,y_train,X_test,y_test)

Train Accuracy: 100.00%
Test Accuracy: 85.25%
ROC score: 0.853
Precision score: 0.87
Recall score: 0.84
F1 Score score: 0.86


SVM

In [28]:
param_dist_svm = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1],
    'kernel': ['linear', 'rbf', 'poly', 'sigmoid']
}

svm_classifier = SVC(random_state=42)
random_search_svm = RandomizedSearchCV(estimator=svm_classifier, param_distributions=param_dist_svm, n_iter=100, cv=5, scoring='recall', random_state=42, n_jobs=-1)

random_search_svm.fit(X_train, y_train)


print("Best parameters found for SVM (Randomized Search): ", random_search_svm.best_params_)
print("Best recall score found for SVM (Randomized Search): ", random_search_svm.best_score_)

Best parameters found for SVM (Randomized Search):  {'kernel': 'sigmoid', 'gamma': 'auto', 'C': 100}
Best recall score found for SVM (Randomized Search):  0.7649350649350649


In [29]:
svm_rs = SVC(C=100, gamma='auto', kernel='sigmoid')
print_performance(svm_rs,X_train,y_train,X_test,y_test)

Train Accuracy: 74.38%
Test Accuracy: 78.69%
ROC score: 0.789
Precision score: 0.83
Recall score: 0.75
F1 Score score: 0.79


**Performance** **Comparison**

In [32]:
results = []

def capture_performance(model, X_train, y_train, X_test, y_test, model_name, search_method):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results.append({
        'Model': model_name,
        'Search Method': search_method,
        'Accuracy': accuracy,
        'ROC AUC': roc_auc,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1
    })


# Capture performance for GridSearchCV models
capture_performance(b_lr, X_train, y_train, X_test, y_test, 'Logistic Regression', 'GridSearchCV')
capture_performance(grid_search_dt.best_estimator_, X_train, y_train, X_test, y_test, 'Decision Tree', 'GridSearchCV')
capture_performance(grid_search_rf.best_estimator_, X_train, y_train, X_test, y_test, 'Random Forest', 'GridSearchCV')
capture_performance(grid_search_svm.best_estimator_, X_train, y_train, X_test, y_test, 'SVM', 'GridSearchCV')

# Capture performance for RandomizedSearchCV models
capture_performance(random_search_lr.best_estimator_, X_train, y_train, X_test, y_test, 'Logistic Regression', 'RandomizedSearchCV')
capture_performance(random_search_dt.best_estimator_, X_train, y_train, X_test, y_test, 'Decision Tree', 'RandomizedSearchCV')
capture_performance(random_search_rf.best_estimator_, X_train, y_train, X_test, y_test, 'Random Forest', 'RandomizedSearchCV')
capture_performance(random_search_svm.best_estimator_, X_train, y_train, X_test, y_test, 'SVM', 'RandomizedSearchCV')

# Create a DataFrame from the results
results_df = pd.DataFrame(results)

# Display the DataFrame
print("Performance Comparison of Tuned Models:")
display(results_df)

Performance Comparison of Tuned Models:


,Model,Search Method,Accuracy,ROC AUC,Precision,Recall,F1 Score
0,Logistic Regression,GridSearchCV,0.885246,0.885776,0.903226,0.87500,0.888889
1,Decision Tree,GridSearchCV,0.770492,0.774784,0.846154,0.68750,0.758621
2,Random Forest,GridSearchCV,0.868852,0.870151,0.900000,0.84375,0.870968
3,SVM,GridSearchCV,0.786885,0.788793,0.827586,0.75000,0.786885
4,Logistic Regression,RandomizedSearchCV,0.836066,0.838901,0.892857,0.78125,0.833333
5,Decision Tree,RandomizedSearchCV,0.868852,0.870151,0.900000,0.84375,0.870968
6,Random Forest,RandomizedSearchCV,0.868852,0.870151,0.900000,0.84375,0.870968
7,SVM,RandomizedSearchCV,0.786885,0.788793,0.827586,0.75000,0.786885


In [36]:
best_model = grid_search.best_estimator_
filename = 'best_logistic_regression_model.pkl'
joblib.dump(best_model, filename)

print(f"Best model saved to {filename}")

Best model saved to best_logistic_regression_model.pkl
